# 82514 · Sesión S19 — SLAM: EKF-SLAM, el grafo de poses y la rejilla de ocupación

**Bloque 6** · lunes 23 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

El mapa y la pose a la vez: EKF-SLAM, el grafo de poses y la rejilla de Nav2.

| | |
|---|---|
| **En clase** | Al acabar el S19a, abrid este y ejecutad la primera celda. En la diapositiva 83, cinco minutos: secciones 0 y 1 sin deteneros; la 4 y la 5; y «✎ Vuestra respuesta». |
| **En casa** | Secciones 2, 3 y 6 y ejercicios; soluciones al final. |
| **Entrega** | Al acabar la clase, individual: este cuaderno y el S19a, ejecutados y con «✎ Vuestra respuesta» escrita (Archivo → Descargar → Descargar .ipynb), a la Tarea «Taller S19: partículas y SLAM» de Moodle. Sin nota: presentado o no presentado. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib')])

import time
import numpy as np, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
from matplotlib.patches import Ellipse, Patch
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares
from scipy.stats import chi2

FIJAS = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0'}   # las de Colab con las que se probó el bloque 6
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

for nombre, mod in (('numpy', np), ('scipy', scipy), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

## 0. El almacén, el robot y el LiDAR

*En clase, sin deteneros · diapositiva 79*

El simulador del S18b, con el LiDAR de S6.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo · el simulador: ejecutadlo sin abrirlo
# El simulador: el de S18b, más el LiDAR, EKF-SLAM, el grafo de poses y la rejilla. Sin detenerse.
import numpy as np
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares

D2R = np.pi / 180
RES, FILAS, COLS = 0.10, 90, 130            # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de S22 (True = ocupado). Fila = y/0,1; columna = x/0,1."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


OCC = mapa_almacen()
MUELLE, PICKING = np.array([0.6, 0.6]), np.array([8.3, 8.2])
BALIZAS = np.array([[0.1, 2.0], [3.0, 0.1], [8.0, 0.1], [12.9, 2.0], [5.5, 4.15],      # nave de abajo
                    [0.1, 7.0], [4.7, 8.9], [10.5, 8.9], [12.9, 6.5], [7.0, 4.75]])    # nave de arriba
RUTA = np.array([[0.6, 0.6], [1.8, 0.6], [1.8, 5.2], [9.6, 5.2], [9.6, 0.6], [0.6, 0.6]])   # la ruta del inventario

V, HZ = 0.5, 10                              # 0,5 m/s; odometria a 10 Hz
DD = V / HZ                                  # 5 cm por paso
SIG_D, SIG_T = 0.06 / np.sqrt(20), 0.2 * D2R # por paso: 13,4 mm y 0,2° (6 cm y 0,9° por metro)
Q = np.diag([SIG_D**2, SIG_T**2])            # Q: la covarianza del ruido de proceso (la odometria)
SIG_R, SIG_PHI, ALCANCE = 0.05, 1.0 * D2R, 8.0
R = np.diag([SIG_R**2, SIG_PHI**2])          # R: la covarianza del ruido de medida (el sensor de balizas)
CADA = 10                                    # un barrido de balizas cada 10 pasos: una vez por segundo
K95 = 2.4477                                 # en el plano, la elipse de 2,45σ encierra el 95 %


def envuelve(a):
    """Un angulo en (−π, π]: sin esto, la innovacion del rumbo salta 360°."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def paso(x, d):
    """El modelo de movimiento: la pose x = (x; y; θ) compuesta con la odometria d = (δ_d; δ_θ) en SE(2),
    como en S8: avanzar δ_d en la direccion del rumbo y despues girar δ_θ."""
    return np.array([x[0] + d[0] * np.cos(x[2]), x[1] + d[0] * np.sin(x[2]), envuelve(x[2] + d[1])])


def jac_paso(x, d):
    """Sus jacobianos: F_x respecto a la pose (3 x 3) y F_v respecto a la odometria (3 x 2)."""
    c, s = np.cos(x[2]), np.sin(x[2])
    Fx = np.array([[1.0, 0.0, -d[0] * s], [0.0, 1.0, d[0] * c], [0.0, 0.0, 1.0]])
    Fv = np.array([[c, 0.0], [s, 0.0], [0.0, 1.0]])
    return Fx, Fv


def h(x, b):
    """El modelo de medida: distancia y rumbo a la baliza b desde la pose x."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    return np.array([np.hypot(dx, dy), envuelve(np.arctan2(dy, dx) - x[2])])


def jac_h(x, b):
    """Su jacobiano H_x respecto a la pose (2 x 3)."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2)
    return np.array([[-dx / r, -dy / r, 0.0], [dy / r2, -dx / r2, -1.0]])


def libre_vista(a, b):
    """¿Ve el sensor la baliza b desde a? Sin celdas ocupadas en medio (la baliza esta en la pared)."""
    n = int(np.ceil(np.linalg.norm(b - a) / 0.02))
    p = a + np.linspace(0, 1, n)[:, None] * (b - a)
    cc = np.clip((p[:, 0] / RES).astype(int), 0, COLS - 1)
    ff = np.clip((p[:, 1] / RES).astype(int), 0, FILAS - 1)
    return not OCC[ff, cc][:-6].any()


def ruta_densa():
    """La ruta del inventario, en pasos de 5 cm."""
    pts = [RUTA[0]]
    for a, b in zip(RUTA[:-1], RUTA[1:]):
        n = int(round(np.linalg.norm(b - a) / DD))
        pts += [a + (b - a) * k / n for k in range(1, n + 1)]
    return np.array(pts)


def mandos(rd):
    """La odometria verdadera de cada paso: el avance y el giro que llevan de un punto de la ruta al siguiente."""
    d, th = [], 0.0
    for a, b in zip(rd[:-1], rd[1:]):
        thn = np.arctan2(b[1] - a[1], b[0] - a[0])
        d.append((np.linalg.norm(b - a), envuelve(thn - th)))
        th = thn
    return np.array(d)


RD = ruta_densa()
U = mandos(RD)                               # 544 pasos, 27,2 m
VERDAD = [np.array([RD[0, 0], RD[0, 1], 0.0])]
for d in U:
    VERDAD.append(paso(VERDAD[-1], d))
VERDAD = np.array(VERDAD)
VISIBLES = [[j for j, b in enumerate(BALIZAS) if np.linalg.norm(b - p[:2]) < ALCANCE and libre_vista(p[:2], b)]
            for p in VERDAD]


def simula(semilla=1):
    """Lo que leen los sensores en una pasada por la ruta: la odometria de cada paso y, una vez por segundo,
    la distancia y el rumbo de cada baliza visible (con su numero: la asociacion se supone conocida)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    med = {}
    for k in range(CADA, len(VERDAD), CADA):
        med[k] = [(j, h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI])) for j in VISIBLES[k]]
    return odo, med
# =====================================================================================================
# El LiDAR de S6 en la rejilla y el campo de distancias (secciones 0 y 6)
# =====================================================================================================
ANG = np.radians(np.arange(0, 360, 0.25))    # 1440 haces: 360° con 0,25° de resolucion
RMAX, SIG_L, P_SIN = 10.0, 0.02, 0.02        # alcance de 10 m, 2 cm de ruido, 2 % de rayos sin retorno
HACES = np.arange(0, ANG.size, 80)           # el filtro usa 18 haces, uno cada 20°
DIST = distance_transform_edt(~OCC) * RES    # el campo de verosimilitud: distancia de cada celda al obstaculo
LIBRE = DIST > 0.28                          # donde cabe el centro del robot (radio de 0,28 m, el de S22)
_FL, _CL = np.nonzero(LIBRE)


def rayos(pose, angs, occ=OCC):
    """La distancia al primer obstaculo de cada rayo, marchando a pasos de 1 cm por la rejilla."""
    t = np.arange(0.01, RMAX, 0.01)
    a = pose[2] + angs
    cc = ((pose[0] + np.outer(np.cos(a), t)) / RES).astype(int)
    ff = ((pose[1] + np.outer(np.sin(a), t)) / RES).astype(int)
    fuera = (cc < 0) | (cc >= COLS) | (ff < 0) | (ff >= FILAS)
    choca = fuera | occ[np.clip(ff, 0, FILAS - 1), np.clip(cc, 0, COLS - 1)]
    r = t[choca.argmax(axis=1)]
    r[~choca.any(axis=1)] = RMAX
    return r


def barrido(pose, rng, idx=None):
    """Un barrido del LiDAR desde la pose (los haces idx; todos si no se dan), con su ruido y sus rayos perdidos."""
    a = ANG if idx is None else ANG[idx]
    r = rayos(pose, a) + rng.normal(0, SIG_L, a.size)
    r[rng.random(a.size) < P_SIN] = RMAX
    return np.minimum(r, RMAX)


def relativa(a, b):
    """La pose de b vista desde a, en SE(2) (la ⊖ de S8): adelante, a la izquierda y el giro."""
    c, s = np.cos(a[2]), np.sin(a[2])
    d = np.asarray(b[:2]) - np.asarray(a[:2])
    return np.array([c * d[0] + s * d[1], -s * d[0] + c * d[1], envuelve(b[2] - a[2])])
# =====================================================================================================
# EKF-SLAM: la pose y las balizas en un solo estado, de 3 + 2N componentes (secciones 2 y 3)
# =====================================================================================================
PUERTA_99 = 9.21                             # χ² con 2 grados de libertad: el 99 % de las medidas buenas, por debajo
SIGMA0 = np.diag([0.02**2, 0.02**2, (1.0 * D2R)**2])   # al salir del muelle: 2 cm y 1° (como en S18b)


def ekf_slam(odo, med, cambia=None, asociar=None, puerta=PUERTA_99, x0=None):
    """EKF-SLAM con la ruta del inventario. El estado es (x; y; θ; m1x; m1y; …): cada baliza nueva lo alarga en
    dos. med[k] = [(j, z)]: j es el numero de la baliza (la asociacion, conocida) salvo que asociar=True, y entonces
    se busca la del mapa con menor d² y, si ninguna pasa la puerta, se toma por nueva.
    cambia = (k, j_real, j_usada): en el barrido k, la medida de j_real se atribuye a j_usada (un error).
    Devuelve el estado y la covarianza en cada paso, el orden en que se anadieron las balizas y las d²."""
    x = (VERDAD[0] if x0 is None else np.asarray(x0)).astype(float).copy()
    S_ = SIGMA0.copy()
    orden, hist, D2 = [], [], []
    for k, d in enumerate(odo, start=1):
        n = x.size
        Fx, Fv = jac_paso(x[:3], d)                           # prediccion: solo se mueve la pose
        x[:3] = paso(x[:3], d)
        F = np.eye(n); F[:3, :3] = Fx
        S_ = F @ S_ @ F.T
        S_[:3, :3] += Fv @ Q @ Fv.T
        for j, z in med.get(k, []):
            if cambia and k == cambia[0] and j == cambia[1]:
                j = cambia[2]
            if asociar:
                j = None
                mejor = (np.inf, None)
                for i, ji in enumerate(orden):
                    nu, Hf, Sm = _innov(x, S_, i, z)
                    d2 = float(nu @ np.linalg.solve(Sm, nu))
                    if d2 < mejor[0]:
                        mejor = (d2, ji)
                if mejor[0] < puerta:
                    j = mejor[1]
                else:
                    j = ('nueva', len(orden))
            if j not in orden:                                # baliza nueva: se anade al estado
                th = x[2] + z[1]
                m = x[:2] + z[0] * np.array([np.cos(th), np.sin(th)])
                Gx = np.array([[1, 0, -z[0] * np.sin(th)], [0, 1, z[0] * np.cos(th)]])
                Gz = np.array([[np.cos(th), -z[0] * np.sin(th)], [np.sin(th), z[0] * np.cos(th)]])
                Sxx = S_[:3, :3]
                Smm = Gx @ Sxx @ Gx.T + Gz @ R @ Gz.T
                Smx = Gx @ S_[:3, :]
                S_ = np.block([[S_, Smx.T], [Smx, Smm]])
                x = np.concatenate([x, m])
                orden.append(j)
                continue
            i = orden.index(j)
            nu, Hf, Sm = _innov(x, S_, i, z)
            D2.append((k, j, float(nu @ np.linalg.solve(Sm, nu))))
            L = S_ @ Hf.T @ np.linalg.inv(Sm)                 # la ganancia, ahora de (3 + 2N) x 2
            x = x + L @ nu
            x[2] = envuelve(x[2])
            S_ = (np.eye(x.size) - L @ Hf) @ S_
        hist.append((x.copy(), S_.copy()))
    return hist, orden, D2


def _innov(x, S_, i, z):
    """La innovacion de la baliza i del estado, su jacobiano respecto a todo el estado y su covarianza."""
    b = x[3 + 2 * i: 5 + 2 * i]
    Hx = jac_h(x[:3], b)
    Hf = np.zeros((2, x.size))
    Hf[:, :3] = Hx
    Hf[:, 3 + 2 * i: 5 + 2 * i] = -Hx[:, :2]
    nu = z - h(x[:3], b)
    nu[1] = envuelve(nu[1])
    return nu, Hf, Hf @ S_ @ Hf.T + R


def error_mapa(x, orden):
    """Error medio y maximo de las balizas del mapa frente a las de verdad (solo las de numero conocido)."""
    n = (x.size - 3) // 2
    e = [np.hypot(*(x[3 + 2 * i: 5 + 2 * i] - BALIZAS[j])) for i, j in enumerate(orden[:n])
         if isinstance(j, (int, np.integer))]
    return (float(np.mean(e)), float(np.max(e))) if e else (np.nan, np.nan)
# =====================================================================================================
# El grafo de poses: un nodo por metro, una arista por tramo de odometria y el cierre de bucle (secciones 4 y 5)
# =====================================================================================================
def odometria_integrada(odo, x0=None):
    """La pose que da la odometria sola, paso a paso (a estima)."""
    X = [VERDAD[0].copy() if x0 is None else np.asarray(x0, float)]
    for d in odo:
        X.append(paso(X[-1], d))
    return np.array(X)


CADA_NODO = 20                               # un nodo cada 20 pasos: cada metro de ruta
# la arista de odometria, isotropa por sencillez: de lado, la odometria da 0,9 cm por metro
S_ARISTA = np.array([SIG_D * np.sqrt(CADA_NODO), SIG_D * np.sqrt(CADA_NODO), SIG_T * np.sqrt(CADA_NODO)])
S_CIERRE = np.array([0.02, 0.02, 0.5 * D2R]) # el cierre: el barrido de ahora alineado con el del principio


def nodos_de(X):
    """Los indices de los nodos (cada metro y el ultimo paso) y sus poses."""
    idx = list(range(0, len(X), CADA_NODO))
    if idx[-1] != len(X) - 1:
        idx.append(len(X) - 1)
    return idx, X[idx]


def pasada_grafo(semilla=20):
    """Una pasada por la ruta para el grafo: la odometria de cada paso y la medida del cierre al volver al
    muelle (la pose del ultimo nodo vista desde el primero, con 2 cm y 0,5° de error)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    idx, Nv = nodos_de(VERDAD)
    cierre = relativa(Nv[0], Nv[-1]) + rng.normal(0, S_CIERRE)
    return odo, (0, len(idx) - 1, cierre)


def grafo(X_odo, cierres=()):
    """Arma el grafo con las poses de la odometria: aristas de odometria entre nodos seguidos y las de cierre
    (i, j, medida). Resuelve min Σ eᵀ·Ω·e con el LM de S10 (least_squares, method='lm'). El primer nodo se fija."""
    idx, N = nodos_de(X_odo)
    aristas = [(i, i + 1, relativa(N[i], N[i + 1]), S_ARISTA) for i in range(len(idx) - 1)]
    aristas += [(i, j, m, S_CIERRE) for i, j, m in cierres]

    def residuos(p):
        Xn = np.vstack([N[0], p.reshape(-1, 3)])
        r = []
        for i, j, m, s in aristas:
            e = relativa(Xn[i], Xn[j]) - m
            e[2] = envuelve(e[2])
            r.append(e / s)
        return np.concatenate(r)

    sol = least_squares(residuos, N[1:].ravel(), method="lm")
    Xg = np.vstack([N[0], sol.x.reshape(-1, 3)])
    r = residuos(sol.x).reshape(-1, 3)
    return dict(idx=idx, N_odo=N, N=Xg, aristas=aristas, chi2=(r**2).sum(axis=1), sol=sol)
# =====================================================================================================
# La rejilla de ocupacion en log-odds, con las poses y el LiDAR (seccion 6)
# =====================================================================================================
L_OCC = np.log(0.7 / 0.3)                    # un impacto: +0,847
L_LIB = np.log(0.3 / 0.7)                    # el rayo pasa por la celda: −0,847
L_TOPE = 5.0                                 # los log-odds se limitan a ±5 (p de 0,007 a 0,993)


def rejilla_ocupacion(poses, barridos, idx=None):
    """Cada celda, una puerta de Bayes binaria en log-odds: el rayo la deja libre hasta el impacto (−0,847 por
    rayo) y la ocupa en el impacto (+0,847). Los rayos sin retorno solo dejan libre."""
    l = np.zeros((FILAS, COLS))
    a0 = ANG if idx is None else ANG[idx]
    for pose, r in zip(poses, barridos):
        a = pose[2] + a0
        for ang, rr in zip(a, r):
            t = np.arange(0.0, rr, 0.05)
            cc = ((pose[0] + t * np.cos(ang)) / RES).astype(int)
            ff = ((pose[1] + t * np.sin(ang)) / RES).astype(int)
            ok = (cc >= 0) & (cc < COLS) & (ff >= 0) & (ff < FILAS)
            cel = np.unique(ff[ok] * COLS + cc[ok])
            if rr < RMAX - 1e-6:
                ci = int((pose[0] + rr * np.cos(ang)) / RES); fi = int((pose[1] + rr * np.sin(ang)) / RES)
                if 0 <= ci < COLS and 0 <= fi < FILAS:
                    imp = fi * COLS + ci
                    cel = cel[cel != imp]
                    l.flat[imp] += L_OCC
            l.flat[cel] += L_LIB
        np.clip(l, -L_TOPE, L_TOPE, out=l)
    return np.round(l, 9)        # un impacto y un paso se anulan: 0 exacto (sin el redondeo, ±10⁻¹⁶ decidiria la celda)


def compara_rejilla(l):
    """La rejilla construida frente al mapa de S22, en las celdas vistas: el % bien clasificadas, y las celdas
    ocupadas fantasma (marcadas ocupadas a mas de 10 cm de cualquier obstaculo verdadero)."""
    vista = l != 0               # p distinta de 0,5: alguna evidencia neta
    ocup = l > 0
    bien = (ocup == OCC) & vista
    fantasma = ocup & (DIST > 0.15)
    return dict(vistas=int(vista.sum()), bien=100 * bien.sum() / vista.sum(),
                ocupadas=int(ocup.sum()), fantasma=int(fantasma.sum()),
                pct_fantasma=100 * fantasma.sum() / max(ocup.sum(), 1))

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
MAPA = ListedColormap(['white', '#4A4F63'])      # el almacén: libre en blanco, ocupado en gris oscuro
TRAZO_VERDAD = dict(color='black', lw=1.8)      # la verdad: una línea negra sólida, como la ruta de esta sección
# para dibujar: un barrido del LiDAR, sin ruido (rayos(): no gasta números aleatorios), desde el pasillo junto al muro
LIDAR_POSE = np.array([6.0, 5.2, 0.0])
LIDAR_PUNTOS = LIDAR_POSE[:2] + rayos(LIDAR_POSE, ANG)[:, None] * np.c_[np.cos(ANG), np.sin(ANG)]

def dibuja_mapa(ax, numeros=False, alfa=0.35):
    """Para dibujar: el almacén (lo ocupado, en gris), las balizas de verdad (cruces verdes) y, con numeros, el número
    de cada una, fuera de la pared; los ejes en metros."""
    ax.imshow(OCC, cmap=MAPA, origin='lower', extent=(0, 13, 0, 9), interpolation='nearest', alpha=alfa)
    ax.plot(BALIZAS[:, 0], BALIZAS[:, 1], 'P', color=IQS_VERDE, ms=10, mec='white', zorder=6)
    if numeros:               # las de las paredes, hacia fuera; las del muro, debajo (la 4) o encima de la ruta (la 9)
        for j, (bx, by) in enumerate(BALIZAS):
            xy, al = (((bx - 0.3, by), dict(ha='right')) if bx < 1 else ((bx + 0.3, by), dict(ha='left')) if bx > 12 else
                      ((bx, by - 0.25), dict(va='top')) if by < 1 else ((bx, by + 0.25), dict(va='bottom')) if by > 8 else
                      ((bx, 3.9), dict(va='top')) if by < 4.45 else ((bx, 5.4), dict(va='bottom')))
            ax.text(*xy, str(j), fontsize=10, fontweight='bold', color=IQS_VERDE, zorder=7,
                    **{'ha': 'center', 'va': 'center', **al}, bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none'))
    lim = 0.7 if numeros else 0.3
    ax.set_xlim(-lim, 13 + lim); ax.set_ylim(-lim, 9 + lim); ax.set_aspect('equal')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ejes(ax)

In [ ]:
# @title ✔ Resultado
print(f'Almacén de {COLS*RES:.0f} x {FILAS*RES:.0f} m ; {len(BALIZAS)} balizas ; ruta de {len(U)} pasos ({coma(len(U)*DD, 1)} m)')
fig, ax = plt.subplots(figsize=(8.6, 6.4))
dibuja_mapa(ax, numeros=True)
ax.fill(LIDAR_PUNTOS[:, 0], LIDAR_PUNTOS[:, 1], color=AMBAR, alpha=0.12, lw=0, zorder=2)
ax.scatter(LIDAR_PUNTOS[:, 0], LIDAR_PUNTOS[:, 1], s=3, color=AMBAR, alpha=0.7, lw=0, zorder=3)
ax.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=2, label='la ruta del inventario (la verdad)')
ax.annotate('', xy=VERDAD[150, :2], xytext=VERDAD[120, :2], arrowprops=dict(arrowstyle='-|>', color='black', lw=2, mutation_scale=18))
ax.plot(*MUELLE, 's', color='black', ms=9, zorder=6, label='el muelle: salida y llegada')
ax.plot(*LIDAR_POSE[:2], 'o', mfc='white', mec='black', mew=1.8, ms=9, zorder=7)
ax.plot([], [], 'P', color=IQS_VERDE, ms=10, label='las 10 balizas, con su número')
ax.plot([], [], 's', color=AMBAR, alpha=0.35, ms=10, label=f'lo que ve el LiDAR desde ({coma(LIDAR_POSE[0], 1)}; {coma(LIDAR_POSE[1], 1)})')
ax.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** 10 balizas y una ruta de 544 pasos, 27,2 m.

## 1. Una pasada por el almacén, sin mapa

*En clase, sin deteneros · diapositiva 79*

Con la odometría sola, la misma baliza cae en sitios distintos.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la semilla de los ruidos de la pasada (la odometría y el sensor de balizas); el grafo de la sección 4 usa la misma
SEMILLA = 20  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
odo, med = simula(SEMILLA)                        # la odometría de cada paso y, cada segundo, las balizas a la vista
Xo = odometria_integrada(odo)                     # la pose con la odometría sola, paso a paso (a estima)
E_ODO = np.hypot(*(Xo[:, :2] - VERDAD[:, :2]).T)  # su error de posición en cada paso [m]
T_PASOS = np.arange(len(VERDAD)) / HZ             # el tiempo de cada paso [s]

def puesta(k, z):
    """Dónde pone la odometría sola una medida (distancia y rumbo) del barrido k."""
    th = Xo[k][2] + z[1]
    return Xo[k][:2] + z[0] * np.array([np.cos(th), np.sin(th)])

# para dibujar: dónde pone cada medida, y la (8,0; 0,1) (la baliza 2) en los pasos 10 y 540, y lo lejos que cae [m]
PUESTAS = np.array([puesta(k, z) for k in sorted(med) for j, z in med[k]])
DOS_SITIOS = {k: puesta(k, dict(med[k])[2]) for k in (10, 540)}
LEJOS = {k: np.hypot(*(p - BALIZAS[2])) for k, p in DOS_SITIOS.items()}

In [ ]:
# @title ✔ Resultado
print(f'la odometría sola: error final {coma(E_ODO[-1]*100, 0)} cm, máximo {coma(E_ODO.max()*100, 0)} cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.0), gridspec_kw={'width_ratios': [1.45, 1]})
dibuja_mapa(a1)
a1.plot(VERDAD[:, 0], VERDAD[:, 1], **TRAZO_VERDAD, label='la ruta (la verdad)')
a1.plot(Xo[:, 0], Xo[:, 1], '--', color=GRIS, lw=1.6, label='la odometría sola')
a1.scatter(PUESTAS[:, 0], PUESTAS[:, 1], s=7, color=AMBAR, alpha=0.7, lw=0, zorder=5, label='donde pone las balizas la odometría')
a1.plot([], [], 'P', color=IQS_VERDE, ms=9, label='las balizas de verdad')
a1.plot([DOS_SITIOS[10][0], DOS_SITIOS[540][0]], [DOS_SITIOS[10][1], DOS_SITIOS[540][1]], color=ROJO, lw=1.2, zorder=7)
for k, y_txt in ((10, -0.55), (540, -1.15)):
    a1.plot(*DOS_SITIOS[k], 'o', ms=12, mfc='none', mec=ROJO, mew=1.8, zorder=8)
    a1.annotate(f'paso {k}: a ' + (f'{coma(LEJOS[k]*100, 0)} cm' if LEJOS[k] < 1 else f'{coma(LEJOS[k], 1)} m'),
                DOS_SITIOS[k], (6.6, y_txt), fontsize=9.5, ha='right', va='center', arrowprops=dict(arrowstyle='-', color=GRIS, lw=1), zorder=9)
a1.set_ylim(-1.45, 9.3)
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
a2.plot(T_PASOS, E_ODO * 100, color=ROJO, lw=1.6)
a2.plot(T_PASOS[-1], E_ODO[-1] * 100, 'o', color=ROJO, ms=6)
a2.annotate(f'al final, {coma(E_ODO[-1]*100, 0)} cm', (T_PASOS[-1], E_ODO[-1] * 100), (T_PASOS[-1] - 1, 10),
            fontsize=10, color=ROJO, ha='right', arrowprops=dict(arrowstyle='-', color=ROJO, lw=1))
a2.set_ylim(0, 1.15 * E_ODO.max() * 100); a2.grid(True, alpha=0.4)
a2.set_xlabel('t (s)'); a2.set_ylabel('error de posición (cm)')
a2.set_title(f'La odometría sola: el error crece hasta {coma(E_ODO.max()*100, 0)} cm', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** 52 cm al final y 97 de máximo; la (8,0; 0,1), en dos sitios a 1,3 m.

## 2. EKF-SLAM: el estado crece con cada baliza

*En casa · diapositiva 80*

Las balizas, dentro del estado. Mirad el paso 270: vuelve a ver la (8,0; 0,1).

Una baliza nueva: m = (x; y) + r·(cos(θ + φ); sen(θ + φ)); sus jacobianos, en ekf_slam.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
t0 = time.time()
hist, orden, D2 = ekf_slam(odo, med)               # EKF-SLAM con la asociación conocida: estado y Σ en cada paso
x, Sigma = hist[-1]                                 # el estado y su covarianza al final
em = error_mapa(x, orden)                           # el error medio y máximo del mapa [m]
ep = np.array([np.hypot(*(h_[0][:2] - VERDAD[k + 1][:2])) for k, h_ in enumerate(hist)])   # el de la pose [m]
T_EKF = time.time() - t0
EM_269_270 = {k: error_mapa(hist[k - 1][0], orden) for k in (269, 270)}   # el mapa antes y después del paso 270
# el mismo barrido del paso 270 sin la medida de la (8,0; 0,1): ¿quién corrige a las dos de arriba a la derecha?
med_sin = dict(med); med_sin[270] = [(j, z) for j, z in med[270] if j != 2]
hist_sin, _, _ = ekf_slam(odo, med_sin)
DOS_ARRIBA = {}       # baliza -> su error en el 269, en el 270 y en el 270 sin esa medida [cm]
for j in (8, 7):
    i = orden.index(j)
    DOS_ARRIBA[j] = [np.hypot(*(hh[3+2*i:5+2*i] - BALIZAS[j])) * 100 for hh in (hist[268][0], hist[269][0], hist_sin[269][0])]
rho = Sigma / np.sqrt(np.outer(np.diag(Sigma), np.diag(Sigma)))   # las correlaciones de Σ
MEMORIA = [(N, 3 + 2 * N) for N in (10, 100, 1000)]               # balizas y componentes del estado
# para dibujar: la pose estimada; el semieje mayor de su σ [cm]; el error de las dos de arriba a la derecha en cada
# paso [cm] (nan antes de entrar en el mapa); y la elipse del 95 % de cada baliza (centro, ejes y ángulo)
X_EKFS = np.array([h_[0][:2] for h_ in hist])
SIG_POSE = np.array([np.sqrt(np.linalg.eigvalsh(h_[1][:2, :2]).max()) for h_ in hist]) * 100
ERR_ARRIBA = {j: np.array([np.hypot(*(h_[0][3+2*orden.index(j):5+2*orden.index(j)] - BALIZAS[j])) * 100
                           if h_[0].size >= 5 + 2 * orden.index(j) else np.nan for h_ in hist]) for j in (8, 7)}
ELIPSES_2 = []
for i in range(len(orden)):
    w, v = np.linalg.eigh(Sigma[3+2*i:5+2*i, 3+2*i:5+2*i])
    ELIPSES_2.append((x[3+2*i:5+2*i], 2*K95*np.sqrt(w[1]), 2*K95*np.sqrt(w[0]), np.degrees(np.arctan2(v[1, 1], v[0, 1]))))

In [ ]:
# @title ✔ Resultado
print(f'{coma(T_EKF, 2)} s ; estado final de {x.size} componentes, Σ de {x.size}x{x.size}')
print(f'mapa: error medio {coma(em[0]*100, 1)} cm, máximo {coma(em[1]*100, 1)} cm ; pose: RMSE {coma(np.sqrt((ep**2).mean())*100, 1)} cm, final {coma(ep[-1]*100, 1)} cm')
for k in (269, 270):
    print(f'paso {k}: mapa, error medio {coma(EM_269_270[k][0]*100, 1)} cm y máximo {coma(EM_269_270[k][1]*100, 1)} cm')
for j in (8, 7):
    e_ = DOS_ARRIBA[j]
    print(f'la ({coma(BALIZAS[j][0], 1)}; {coma(BALIZAS[j][1], 1)}): {coma(e_[0], 1)} cm en el 269, {coma(e_[1], 1)} en el 270 ; sin la medida de la (8,0; 0,1), {coma(e_[2], 1)}')
for N, n in MEMORIA:
    print(f'con {N} balizas: estado de {n}, Σ de {n*n} entradas, {coma(n*n*8/1e6, 2)} MB')
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw={'width_ratios': [1.4, 1.2, 0.8]})
dibuja_mapa(a1)
a1.plot(VERDAD[:, 0], VERDAD[:, 1], **TRAZO_VERDAD, label='la ruta (la verdad)')
a1.plot(X_EKFS[:, 0], X_EKFS[:, 1], color=IQS_AZUL, lw=1.2, label='la pose estimada')
for c_, an, al, ang in ELIPSES_2:
    a1.add_patch(Ellipse(c_, an, al, angle=ang, fc=AMBAR, alpha=0.45, ec=AMBAR, lw=1.2, zorder=5))
    a1.plot(*c_, '.', color=AMBAR, ms=5, zorder=5)
a1.plot([], [], 'P', color=IQS_VERDE, ms=9, label='las balizas de verdad')
h_, l_ = a1.get_legend_handles_labels()
a1.legend(h_ + [Patch(fc=AMBAR, alpha=0.45)], l_ + ['su estimación (elipse del 95 %)'], fontsize=9, loc='lower left',
          bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
t_270 = 270 / HZ
a2.fill_between(T_PASOS[1:], 0, K95 * SIG_POSE, color=IQS_AZUL, alpha=0.15, lw=0, label='la pose: su 2,45σ')
a2.plot(T_PASOS[1:], ep * 100, color=IQS_AZUL, lw=1.6, label='la pose: su error')
for j, ls in ((8, '-'), (7, '--')):
    a2.plot(T_PASOS[1:], ERR_ARRIBA[j], ls, color=AMBAR, lw=1.6, label=f'la baliza ({coma(BALIZAS[j][0], 1)}; {coma(BALIZAS[j][1], 1)})')
y2 = 1.45 * max(np.nanmax(K95 * SIG_POSE), np.nanmax(ERR_ARRIBA[8]))
a2.axvline(t_270, color='black', ls=':', lw=1.2)
a2.text(t_270 + 0.8, 0.68 * y2, 'paso 270: vuelve a\nver la (8,0; 0,1)', fontsize=9.5, va='top')
a2.set_ylim(0, y2)
a2.set_xlabel('t (s)'); a2.set_ylabel('cm'); a2.grid(True, alpha=0.4)
a2.legend(fontsize=9, loc='upper left', ncol=2, columnspacing=0.8, handlelength=1.6)
a2.set_title('El error: la pose y dos balizas, juntas', fontsize=11); ejes(a2)
im = a3.imshow(np.abs(rho), cmap='Blues', vmin=0, vmax=1)
n_b = len(orden)
a3.set_xticks([0, 3, 1 + 2 * n_b]); a3.set_xticklabels(['x', '$m_1$', f'$m_{{{n_b}}}$'])
a3.set_yticks([0, 3, 1 + 2 * n_b]); a3.set_yticklabels(['x', '$m_1$', f'$m_{{{n_b}}}$'])
a3.set_title(f'|correlación| en Σ: {x.size} × {x.size}', fontsize=11)
cb = fig.colorbar(im, ax=a3, fraction=0.046, pad=0.04); cb.ax.yaxis.set_major_formatter(COMA)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** mapa, 9,7 cm; pose, 13,0 cm de RMSE; en el paso 270, dos balizas bajan de 32 y 26 cm a 18.

## 3. ¿Es la baliza de antes? La asociación de datos

*En casa · diapositiva 81*

Una medida atribuida a otra baliza, sin puerta y con ella; y, sin saber cuál es, la de menor d².

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la medida mal asociada: (paso del barrido, baliza de verdad, baliza a la que se atribuye), con los números de
# la sección 0; el barrido del paso 310 es t = 31 s
CAMBIO = (310, 2, 0)  # @param {type:"raw"}
# la puerta de Mahalanobis: se rechazan las medidas con d² por encima (9,21 deja fuera el 1 % de las buenas)
PUERTA = 9.21  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
hb, ob, Db = ekf_slam(odo, med, cambia=CAMBIO)                  # con la medida equivocada, sin puerta
D2_MALA = next((d for k, j, d in Db if k == CAMBIO[0] and j == CAMBIO[2]), np.nan)   # su d² (nan: no se compara)
EM_MALA = error_mapa(hb[-1][0], ob)                               # el mapa al final [m]
d2 = np.array([d[2] for d in D2])                                 # las d² de la sección 2, todas bien asociadas
ESPERADO = 100 * np.exp(-PUERTA / 2)                              # el % de las buenas por encima (χ², 2 g. l.)
FUERA = [(k, d) for k, j, d in D2 if d > PUERTA]                  # las buenas que la puerta rechazaría
# con la puerta: fuera las buenas con d² > PUERTA y, si su d² pasa de PUERTA, la equivocada; si no, entra equivocada
fuera_p = {(k, j) for k, j, d in D2 if d > PUERTA} | ({(CAMBIO[0], CAMBIO[1])} if D2_MALA > PUERTA else set())
hp, op, _ = ekf_slam(odo, {k: [(j, z) for j, z in v if (k, j) not in fuera_p] for k, v in med.items()}, cambia=CAMBIO)
ha, oa, _ = ekf_slam(odo, med, asociar=True, puerta=PUERTA)       # sin saber cuál es: la de menor d², o una nueva
xa = ha[-1][0]
MAPA_ASOC = []        # (baliza del mapa, número de la de verdad más cercana)
for i in range((xa.size - 3) // 2):
    m = xa[3+2*i:5+2*i]
    MAPA_ASOC.append((m, int(np.argmin(np.hypot(*(BALIZAS - m).T)))))
# para dibujar: el error medio del mapa en cada paso [cm]; las balizas de verdad con más de una copia; la pose
EM_PASO = {nom: np.array([error_mapa(h_[0], o_)[0] for h_ in hh]) * 100
           for nom, hh, o_ in (('bien', hist, orden), ('mal', hb, ob), ('puerta', hp, op))}
COPIAS = {j: n for j in range(len(BALIZAS)) if (n := sum(jj == j for _, jj in MAPA_ASOC)) > 1}   # baliza -> copias
X_ASOC = np.array([h_[0][:2] for h_ in ha])

In [ ]:
# @title ✔ Resultado
print(f'una medida mal asociada en el paso {CAMBIO[0]}: su d² = {coma(D2_MALA, 0)} ;'
      f' el mapa acaba con {coma(EM_MALA[0]*100, 0)} cm de error medio (sin el error, {coma(em[0]*100, 1)})')
print(f'las {d2.size} medidas bien asociadas: d² medio {coma(d2.mean())} ; por encima de {coma(PUERTA)}: {len(FUERA)} (se espera el {coma(ESPERADO, 0)} %, {coma(ESPERADO / 100 * d2.size, 1)}):',
      ', '.join(f'paso {k}, d² = {coma(d, 1)}' for k, d in FUERA))
print(f'sin saber cuál es: el mapa acaba con {len(MAPA_ASOC)} balizas')
for m, j in MAPA_ASOC:
    print(f'  ({coma(m[0])}; {coma(m[1])}) -> la de verdad más cercana, ({coma(BALIZAS[j][0], 1)}; {coma(BALIZAS[j][1], 2)})')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={'width_ratios': [1, 1.15]})
a1.plot(T_PASOS[1:], EM_PASO['bien'], color=GRIS, lw=4, alpha=0.5, zorder=1, label='sin equivocarse')
a1.plot(T_PASOS[1:], EM_PASO['mal'], color=ROJO, lw=1.8, label='la equivocada, sin puerta')
a1.plot(T_PASOS[1:], EM_PASO['puerta'], '--', color=IQS_AZUL, lw=1.8,
        label=f'con la puerta de {coma(PUERTA)}: descarta ' + ('la equivocada y ' if D2_MALA > PUERTA else '') + f'{len(FUERA)} buenas')
a1.axvline(CAMBIO[0] / HZ, color='black', ls=':', lw=1.2)
a1.text(CAMBIO[0] / HZ + 0.8, 1.5, f'paso {CAMBIO[0]}: d² = {coma(D2_MALA, 0)}', fontsize=9.5)
a1.set_yscale('log'); a1.set_ylim(1, 1000); a1.yaxis.set_minor_formatter(plt.NullFormatter())
a1.set_xlabel('t (s)'); a1.set_ylabel('error medio del mapa (cm, escala log.)'); a1.grid(True, alpha=0.4)
a1.legend(fontsize=9, loc='upper left'); a1.set_title('Una medida equivocada: el error se queda en el mapa', fontsize=11); ejes(a1)
dibuja_mapa(a2)
a2.plot(VERDAD[:, 0], VERDAD[:, 1], **TRAZO_VERDAD, label='la ruta (la verdad)')
a2.plot(X_ASOC[:, 0], X_ASOC[:, 1], color=IQS_AZUL, lw=1.1, label='la pose estimada')
a2.plot([m[0] for m, _ in MAPA_ASOC], [m[1] for m, _ in MAPA_ASOC], 'o', color=AMBAR, ms=8, mec='#333', zorder=7, label='las balizas del mapa')
for j, n in COPIAS.items():
    a2.plot(*BALIZAS[j], 'o', ms=24, mfc='none', mec=ROJO, mew=2, zorder=8)
    a2.annotate(f'{n} copias', BALIZAS[j], (np.clip(BALIZAS[j][0], 1.5, 11.5) + 0.3, np.clip(BALIZAS[j][1], 1.6, 7.4)), fontsize=10,
                color=ROJO, ha='center', va='center', arrowprops=dict(arrowstyle='->', color=ROJO, lw=1.2, shrinkB=12), zorder=9)
a2.plot([], [], 'P', color=IQS_VERDE, ms=9, label='las de verdad')
a2.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, columnspacing=0.8,
          title=f'Sin saber cuál es: {len(MAPA_ASOC)} balizas en el mapa', title_fontsize=10, alignment='left')
plt.tight_layout(); plt.show()

**Qué tiene que salir:** d² = 11 532 y 217 cm de error en el mapa; sin saber cuál es, 11 balizas.

## 4. El grafo de poses, sin cierre y con él

*En clase · diapositivas 82 y 83*

29 nodos, 28 aristas de odometría y la del cierre; los ajusta el LM de S10.

### En clase: el error de cada nodo

¿Dónde va el error al cerrar? Mirad la figura de la derecha.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# ¿se usa el cierre al volver al muelle (el nodo 28 visto desde el 0)?
CON_CIERRE = True  # @param {type:"boolean"}
# la σ que el grafo da al cierre, en x y en y [cm] y en el giro [°] (el cierre se midió con 2 cm y 0,5°)
SIGMA_CIERRE_CM = 2.0  # @param {type:"number"}
SIGMA_CIERRE_GRADOS = 0.5  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
odoG, cierre = pasada_grafo(SEMILLA)        # la pasada del grafo y el cierre medido (con 2 cm y 0,5°)
XoG = odometria_integrada(odoG)               # las poses de la odometría: de ahí salen los nodos
idx, Nv = nodos_de(VERDAD)                    # los nodos de verdad, uno por metro
s_cierre_sim = S_CIERRE                       # la σ del cierre del simulador: el grafo lee S_CIERRE; luego, como estaba
S_CIERRE = np.array([SIGMA_CIERRE_CM / 100, SIGMA_CIERRE_CM / 100, SIGMA_CIERRE_GRADOS * D2R])
try:
    t0 = time.time()
    sin = grafo(XoG, ())                                 # solo aristas de odometría
    con = grafo(XoG, [cierre] if CON_CIERRE else ())     # y la del cierre
    T_GRAFO = (time.time() - t0) / 2
finally:
    S_CIERRE = s_cierre_sim
e_odo = np.hypot(*(con['N_odo'][:, :2] - Nv[:, :2]).T)  # el error de cada nodo frente a la verdad [m]
e_sin = np.hypot(*(sin['N'][:, :2] - Nv[:, :2]).T)
e_con = np.hypot(*(con['N'][:, :2] - Nv[:, :2]).T)

In [ ]:
# @title ✔ Resultado
print(f'{len(idx)} nodos, {len(con["aristas"])} aristas, {3*(len(idx) - 1)} incógnitas ; {coma(T_GRAFO, 2)} s cada una')
nom_con = 'grafo con el cierre' if CON_CIERRE else 'grafo (sin el cierre)'
for nom, e in (('solo odometría', e_odo), ('grafo sin cierre', e_sin), (nom_con, e_con)):
    print(f'{nom:20s}: final {coma(e[-1]*100, 1)} cm, máximo {coma(e.max()*100, 1)} cm, medio {coma(e.mean()*100, 1)} cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.7), gridspec_kw={'width_ratios': [1.3, 1]})
dibuja_mapa(a1)
a1.plot(Nv[:, 0], Nv[:, 1], **TRAZO_VERDAD, label='la verdad')
a1.plot(*con['N_odo'][:, :2].T, '--o', color=GRIS, lw=1.4, ms=3.5, label='sin cierre (la odometría)')
a1.plot(*con['N'][:, :2].T, '-o', color=IQS_AZUL, lw=1.6, ms=4, label=nom_con)
if CON_CIERRE:
    a1.plot(*np.array([con['N_odo'][0, :2], con['N_odo'][-1, :2]]).T, color=IQS_VERDE, lw=4, zorder=6, label='el cierre: el nodo 28 visto desde el 0')
a1.annotate('nodo 0', con['N_odo'][0, :2], (2.3, 1.5), fontsize=9.5, arrowprops=dict(arrowstyle='-', color=GRIS))
a1.annotate('nodo 28', con['N_odo'][-1, :2], (0.25, 3.0), fontsize=9.5, arrowprops=dict(arrowstyle='-', color=GRIS))
a1.set_ylim(min(-0.4, con['N_odo'][:, 1].min() - 0.3), 9.3)
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
n_ = np.arange(len(idx))
a2.plot(n_, e_sin * 100, '--o', color=GRIS, lw=1.4, ms=4, label='sin cierre')
a2.plot(n_, e_con * 100, '-o', color=IQS_AZUL, lw=1.6, ms=4, label=nom_con)
a2.axvline(n_[-1], color=IQS_VERDE, lw=2.5, alpha=0.8)
a2.text(n_[-1] - 0.4, 0.97 * 1.12 * e_sin.max() * 100, 'el nodo\ndel cierre', fontsize=9.5, color=IQS_VERDE, ha='right', va='top')
a2.set_ylim(0, 1.12 * e_sin.max() * 100); a2.set_xlim(-0.5, n_[-1] + 0.8); a2.grid(True, alpha=0.4)
a2.set_xlabel('nodo (uno por metro)'); a2.set_ylabel('error frente a la verdad (cm)'); a2.legend(fontsize=9, loc='upper left')
a2.set_title(f'El error de cada nodo: el máximo, de {coma(e_sin.max()*100, 0)} a {coma(e_con.max()*100, 0)} cm', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** sin cierre, 52 cm al final y 94 de máximo; con él, 2,9 y 27,3 cm.

## 5. Un cierre falso

*En clase · diapositivas 83 y 84*

El pasillo de arriba hacia el este (nodo 10) y el de abajo hacia el oeste (nodo 21), tomados por el mismo sitio.

### En clase: los nodos 10 y 21

¿Qué le pasa al mapa? ¿Y a los residuos, eᵀ·Ω·e? Probad otro par.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los dos nodos que el cierre falso dice que son el mismo sitio, girado 180° (de 0 a 28)
NODO_A = 10  # @param {type:"integer"}
NODO_B = 21  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
m_falsa = np.array([0.0, 0.0, np.pi])                          # «el mismo sitio, girado 180°»
bueno = grafo(XoG, [cierre])                                   # solo el cierre bueno (2 cm y 0,5°)
falso = grafo(XoG, [cierre, (NODO_A, NODO_B, m_falsa)])        # el bueno y el falso
sin_bueno = grafo(XoG, [(NODO_A, NODO_B, m_falsa)])            # solo el falso
e_f = np.hypot(*(falso['N'][:, :2] - Nv[:, :2]).T)             # el error de cada nodo con el falso [m]
sobran = lambda g: 3 * len(g['aristas']) - 3 * (len(g['idx']) - 1)   # residuos − incógnitas
SUMAS = [(nom, g['chi2'].sum(), sobran(g), chi2.ppf(0.99, sobran(g)))  # la suma y su umbral del 99 % (χ²)
         for nom, g in (('con el cierre bueno', bueno), ('con el bueno y el falso', falso))]

In [ ]:
# @title ✔ Resultado
print(f'nodos {NODO_A} y {NODO_B} ; con el cierre falso: error máximo {coma(e_f.max())} m, final {coma(e_f[-1]*100, 1)} cm')
for nom, suma, sob, u99 in SUMAS:
    print(f"{nom}: suma {coma(suma, 1)} ; sobran {sob} residuos ; el 99 % queda por debajo de {coma(u99, 1)}")
print(f"la arista falsa sola: {coma(falso['chi2'][-1], 1)} ; las de odometría: {coma(falso['chi2'][:-2].mean(), 1)} de media")
print(f"quitando el falso: {coma(bueno['chi2'].sum(), 1)} ; quitando el bueno: {coma(sin_bueno['chi2'].sum(), 0)}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.7), gridspec_kw={'width_ratios': [1.15, 1]})
dibuja_mapa(a1)
a1.plot(Nv[:, 0], Nv[:, 1], **TRAZO_VERDAD, label='la verdad')
a1.plot(*bueno['N'][:, :2].T, '-o', color=IQS_AZUL, lw=1.4, ms=3.5, label='con el cierre bueno')
a1.plot(*falso['N'][:, :2].T, '-o', color=ROJO, lw=1.6, ms=3.5, label='con el bueno y el falso')
a1.plot(*Nv[[NODO_A, NODO_B], :2].T, '--o', color=AMBAR, lw=2.5, ms=7, zorder=6, label=f'el cierre falso une los nodos {NODO_A} y {NODO_B}')
a1.plot(*falso['N'][[NODO_A, NODO_B], :2].T, 'o', ms=13, mfc='none', mec=AMBAR, mew=2.2, zorder=7, label='y el grafo los junta')
for n_, dy in ((NODO_A, 0.35), (NODO_B, -0.35)):
    a1.text(Nv[n_, 0], Nv[n_, 1] + dy, f'nodo {n_}', fontsize=9.5, color=AMBAR, ha='center', va='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.85), zorder=8)
todos = np.vstack([falso['N'][:, :2], bueno['N'][:, :2]])
a1.set_xlim(min(-0.3, todos[:, 0].min() - 0.4), max(13.3, todos[:, 0].max() + 0.4))
a1.set_ylim(min(-0.3, todos[:, 1].min() - 0.4), max(9.3, todos[:, 1].max() + 0.4))
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
no = len(bueno['chi2']) - 1                     # aristas de odometría; los cierres, aparte: el bueno en no + 1,5 y el falso en no + 4,5
xb, xf = np.r_[np.arange(no), no + 1.5], np.r_[np.arange(no), no + 1.5, no + 4.5]
a2.bar(xb - 0.2, bueno['chi2'], 0.4, color=IQS_AZUL, label=f"con el bueno: suma {coma(SUMAS[0][1], 1)}")
bb = a2.bar(xf + 0.2, falso['chi2'], 0.4, color=ROJO, label=f"con el bueno y el falso: suma {coma(SUMAS[1][1], 0)}")
bb[-1].set_color(AMBAR); bb[-1].set_edgecolor('black')
a2.annotate(f"el falso: {coma(falso['chi2'][-1], 1)}", (xf[-1] + 0.2, falso['chi2'][-1]), (xf[-1] - 5, 300), fontsize=9.5, ha='right',
            arrowprops=dict(arrowstyle='->', color='black', lw=1))
a2.set_yscale('log'); a2.set_ylim(1e-3, 3e3); a2.yaxis.set_minor_formatter(plt.NullFormatter()); a2.set_xlim(-1, xf[-1] + 1.2)
a2.set_xticks(list(range(0, no, 5)) + [no + 1.5, no + 4.5]); a2.set_xticklabels([str(i) for i in range(0, no, 5)] + ['bueno', 'falso'])
a2.set_xlabel('arista: de odometría (de cada nodo al siguiente) y los cierres'); a2.set_ylabel('eᵀ·Ω·e (escala log.)')
a2.legend(fontsize=9, loc='upper left'); a2.grid(True, axis='y', alpha=0.4); a2.set_axisbelow(True)
a2.set_title('El residuo de cada arista, con el cierre bueno y con el falso', fontsize=11); a2.yaxis.set_major_formatter(COMA)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** hasta 2,70 m de error; la arista falsa, 10,9; la suma, de 7,1 (umbral 11,3) a 761 (umbral 16,8): antes de aceptar un cierre, mirad la suma.

### ✎ Vuestra respuesta (dos líneas)

*Doble clic aquí y escribid, con cifras. Al acabar la clase, Archivo → Descargar → Descargar .ipynb de los dos cuadernos, el S19a y el S19b, y a la Tarea «Taller S19: partículas y SLAM»: cada uno, los suyos.*

1. El grafo sin cierre y con él: el error de cada nodo frente a la verdad; ¿dónde va el error al cerrar?
2. Un cierre falso entre dos pasillos que el LiDAR ve iguales (el de arriba hacia el este y el de abajo hacia el oeste): ¿qué le pasa al mapa? ¿Y a los residuos?

## 6. La rejilla de ocupación: el mapa de S22

*En casa · diapositiva 85*

Una puerta de Bayes por celda, en log-odds, con tres juegos de poses.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la p de «ocupada» que deja un impacto; un rayo que pasa deja 1 − p (0,7: ±0,847 en log-odds)
P_IMPACTO = 0.7  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
p_libre = round(1 - P_IMPACTO, 12)                  # 1 − p, sin el 0,30000000000000004 de la resta
L_OCC, L_LIB = np.log(P_IMPACTO / p_libre), np.log(p_libre / P_IMPACTO)   # lo que suma un impacto y resta un rayo
rng = np.random.default_rng(55)                     # el ruido del LiDAR, con su propia semilla
barridos = [barrido(p, rng) for p in Nv]            # lo que ve el LiDAR desde cada nodo de verdad
sub = np.arange(0, ANG.size, 2)                     # uno de cada dos haces: 720
REJILLAS = []         # (nombre, log-odds, comparación con el mapa de S22, para dibujar: 1 − p y las celdas fantasma)
for nom, poses in (('poses de la odometría', bueno['N_odo']), ('poses del grafo', bueno['N']), ('poses verdaderas', Nv)):
    l = rejilla_ocupacion(poses, [b[sub] for b in barridos], idx=sub)
    REJILLAS.append((nom, l, compara_rejilla(l), 1 / (1 + np.exp(l)), (l > 0) & (DIST > 0.15)))

In [ ]:
# @title ✔ Resultado
for nom, l, cmp_, _, _ in REJILLAS:
    print(f"{nom:22s}: bien clasificadas {coma(cmp_['bien'], 1)} % ; ocupadas {cmp_['ocupadas']}, a más de 10 cm de una pared de verdad {cmp_['fantasma']}")
fig, axs = plt.subplots(1, 3, figsize=(15, 4.0))
for ax, (nom, l, cmp_, libre, fant) in zip(axs, REJILLAS):
    ax.imshow(libre, cmap='gray', origin='lower', extent=(0, 13, 0, 9), vmin=0, vmax=1, interpolation='nearest')
    ax.imshow(np.ma.masked_where(~fant, fant), cmap=ListedColormap([ROJO]), origin='lower', extent=(0, 13, 0, 9), interpolation='nearest')
    ax.set_title(f"Con las {nom}: {coma(cmp_['pct_fantasma'], 0)} % fantasma", fontsize=11)
    ax.set_xlabel('x (m)'); ejes(ax)
axs[0].set_ylabel('y (m)')
fig.legend(handles=[Patch(fc='black', label='ocupada'), Patch(fc='white', ec='#999', label='libre'), Patch(fc='#808080', label='sin ver (p = 0,5)'),
                    Patch(fc=ROJO, label='ocupada a más de 10 cm de una pared: fantasma')], fontsize=9, loc='lower center', ncol=4, frameon=False)
plt.tight_layout(rect=(0, 0.09, 1, 1)); plt.show()

**Qué tiene que salir:** paredes fantasma, el 66 % con la odometría, el 11 % con el grafo y el 0 % con la verdad.

### Ejercicio 1 (en casa): balizas de corto alcance

Solo las medidas de menos de 4 m: ¿qué les pasa al mapa y a la pose?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el alcance del sensor de balizas [m] (en la sección 2, 8 m)
E1_ALCANCE = 4.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E1_MED = {k: [(j, z) for j, z in v if z[0] < E1_ALCANCE] for k, v in med.items()}   # las de menos de E1_ALCANCE
e1_hist, e1_orden, _ = ekf_slam(odo, E1_MED)
E1_N = (sum(len(v) for v in med.values()), sum(len(v) for v in E1_MED.values()), sum(1 for v in E1_MED.values() if not v))
E1_EM = error_mapa(e1_hist[-1][0], e1_orden)        # el error medio y máximo del mapa [m]
E1_EP = np.array([np.hypot(*(h_[0][:2] - VERDAD[k + 1][:2])) for k, h_ in enumerate(e1_hist)])   # el de la pose [m]
# para dibujar: el error final de cada baliza [cm], con 8 m (sección 2) y con E1_ALCANCE (nan si no entra en el mapa)
E1_BAL = np.full((2, len(BALIZAS)), np.nan)
for e1_f, (e1_x, e1_o) in enumerate(((x, orden), (e1_hist[-1][0], e1_orden))):
    for e1_i, e1_j in enumerate(e1_o):
        E1_BAL[e1_f, e1_j] = np.hypot(*(e1_x[3+2*e1_i:5+2*e1_i] - BALIZAS[e1_j])) * 100

In [ ]:
# @title ✔ Resultado
print(f'con {coma(E1_ALCANCE, 1)} m: quedan {E1_N[1]} medidas de {E1_N[0]} ; barridos sin ninguna, {E1_N[2]} ; balizas en el mapa, {len(e1_orden)}')
print(f'mapa: error medio {coma(E1_EM[0]*100, 1)} cm y máximo {coma(E1_EM[1]*100, 1)} (con 8 m: {coma(em[0]*100, 1)} y {coma(em[1]*100, 1)})')
print(f'pose: RMSE {coma(np.sqrt((E1_EP**2).mean())*100, 1)} cm y máximo {coma(E1_EP.max()*100, 1)} (con 8 m: {coma(np.sqrt((ep**2).mean())*100, 1)} y {coma(ep.max()*100, 1)})')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 3.9), gridspec_kw={'width_ratios': [1.3, 1]})
for err, col, t in ((ep, IQS_AZUL, 'hasta 8 m'), (E1_EP, AMBAR, f'hasta {coma(E1_ALCANCE, 1)} m')):
    a1.plot(T_PASOS[1:], err * 100, color=col, lw=1.6, label=t)
a1.set_xlabel('t (s)'); a1.set_ylabel('cm'); a1.grid(True, alpha=0.4); a1.legend(fontsize=9, loc='upper left')
a1.set_title('El error de la pose', fontsize=11); ejes(a1)
xb = np.arange(len(BALIZAS))
a2.bar(xb - 0.2, E1_BAL[0], 0.4, color=IQS_AZUL, label='hasta 8 m')
a2.bar(xb + 0.2, E1_BAL[1], 0.4, color=AMBAR, label=f'hasta {coma(E1_ALCANCE, 1)} m')
a2.set_xticks(xb); a2.set_xlabel('baliza (su número, el de la sección 0)'); a2.set_ylabel('cm')
a2.grid(True, axis='y', alpha=0.4); a2.set_axisbelow(True); a2.legend(fontsize=9, loc='upper left')
a2.set_title('El error de cada baliza, al final', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

### Ejercicio 2 (en casa): un cierre que se fía poco

Con la σ del cierre 10 y 100 veces mayor: ¿cuánto corrige?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# por cuánto se multiplica la σ del cierre (2 cm y 0,5°)
E2_FACTORES = (10, 100)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e2_sim = S_CIERRE                             # la σ del cierre del simulador: el grafo lee S_CIERRE; luego, como estaba
E2 = []                                       # (factor, nodos del grafo, error de cada nodo [m])
try:
    for e2_f in E2_FACTORES:
        S_CIERRE = e2_sim * e2_f
        e2_g = grafo(XoG, [cierre])
        E2.append((e2_f, e2_g['N'], np.hypot(*(e2_g['N'][:, :2] - Nv[:, :2]).T)))
finally:
    S_CIERRE = e2_sim
E2_E_UNO = np.hypot(*(bueno['N'][:, :2] - Nv[:, :2]).T)   # para dibujar: con la σ de verdad (la sección 5) [m]

In [ ]:
# @title ✔ Resultado
for f, _, e in E2:
    print(f'σ del cierre ×{f} ({coma(2*f, 0)} cm y {0.5*f:g}°'.replace('.', ',') + f'): final {coma(e[-1]*100, 1)} cm, máximo {coma(e.max()*100, 1)} cm, medio {coma(e.mean()*100, 1)} cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios': [1.15, 1]})
dibuja_mapa(a1)
a1.plot(Nv[:, 0], Nv[:, 1], **TRAZO_VERDAD, label='la verdad')
a1.plot(*bueno['N_odo'][:, :2].T, '--', color=GRIS, lw=1.3, label='sin cierre')
a1.plot(*bueno['N'][:, :2].T, '-', color=IQS_AZUL, lw=1.6, label='σ ×1: 2 cm y 0,5°')
a2.plot(e_odo * 100, '--o', color=GRIS, lw=1.3, ms=3.5, label='sin cierre')
a2.plot(E2_E_UNO * 100, '-o', color=IQS_AZUL, lw=1.6, ms=3.5, label='σ ×1')
for (f, N_, e), col in zip(E2, (AMBAR, '#7D3C98', '#117A65', '#6E2C00') * 3):        # una variante, un color
    a1.plot(*N_[:, :2].T, '-', color=col, lw=1.6, label=f'σ ×{f}')
    a2.plot(e * 100, '-o', color=col, lw=1.6, ms=3.5, label=f'σ ×{f}')
a1.set_ylim(-0.7, 9.7); a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, frameon=False)
a2.set_xlabel('nodo (uno por metro)'); a2.set_ylabel('error frente a la verdad (cm)'); a2.grid(True, alpha=0.4)
a2.legend(fontsize=9, loc='upper left', ncol=2); a2.set_title('El error de cada nodo, con cada σ del cierre', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

### Ejercicio 3 (en casa): una rejilla más prudente

Con p = 0,6 por impacto: ¿cambian las paredes fantasma? ¿Qué las arregla?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la p de «ocupada» que deja un impacto (en la sección 6, 0,7)
E3_P_IMPACTO = 0.6  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e3_antes = (L_OCC, L_LIB)                           # los de la sección 6: la rejilla los lee; luego, como estaban
e3_libre = round(1 - E3_P_IMPACTO, 12)
L_OCC, L_LIB = np.log(E3_P_IMPACTO / e3_libre), np.log(e3_libre / E3_P_IMPACTO)
E3 = []               # (nombre, comparación con el mapa de S22, para dibujar: 1 − p y las celdas fantasma)
try:
    for e3_nom, e3_poses in (('poses de la odometría', bueno['N_odo']), ('poses del grafo', bueno['N'])):
        e3_l = rejilla_ocupacion(e3_poses, [b[sub] for b in barridos], idx=sub)
        E3.append((e3_nom, compara_rejilla(e3_l), 1 / (1 + np.exp(e3_l)), (e3_l > 0) & (DIST > 0.15)))
finally:
    L_OCC, L_LIB = e3_antes

In [ ]:
# @title ✔ Resultado
for (nom, c3, _, _), (_, _, c6, _, _) in zip(E3, REJILLAS):
    print(f"{nom}: con p = {coma(E3_P_IMPACTO, 1)}, el {coma(c3['pct_fantasma'], 0)} % fantasma ({c3['fantasma']} de {c3['ocupadas']}), {coma(c3['bien'], 1)} % bien ;"
          f" con {coma(P_IMPACTO, 1)}, el {coma(c6['pct_fantasma'], 0)} % ({c6['fantasma']} de {c6['ocupadas']}), {coma(c6['bien'], 1)} %")
fig, axs = plt.subplots(1, 3, figsize=(15, 3.9), gridspec_kw={'width_ratios': [1, 1, 0.8]})
for ax, (nom, c3, libre, fant) in zip(axs, E3):
    ax.imshow(libre, cmap='gray', origin='lower', extent=(0, 13, 0, 9), vmin=0, vmax=1, interpolation='nearest')
    ax.imshow(np.ma.masked_where(~fant, fant), cmap=ListedColormap([ROJO]), origin='lower', extent=(0, 13, 0, 9), interpolation='nearest')
    ax.set_title(f"p = {coma(E3_P_IMPACTO, 1)}, {nom}: {coma(c3['pct_fantasma'], 0)} % fantasma", fontsize=10.5)
    ax.set_xlabel('x (m)'); ejes(ax)
axs[0].set_ylabel('y (m)')
xg = np.arange(len(E3))
for k_, (p_, cs, col) in enumerate(((P_IMPACTO, [r[2] for r in REJILLAS[:len(E3)]], GRIS), (E3_P_IMPACTO, [r[1] for r in E3], AMBAR))):
    bb = axs[2].bar(xg + (k_ - 0.5) * 0.38, [c['pct_fantasma'] for c in cs], 0.36, color=col, label=f'p = {coma(p_, 1)}')
    axs[2].bar_label(bb, labels=[coma(c['pct_fantasma'], 0) + ' %' for c in cs], fontsize=9.5)
axs[2].set_xticks(xg); axs[2].set_xticklabels(['odometría', 'grafo']); axs[2].set_ylim(0, 85); axs[2].set_ylabel('% de lo ocupado')
axs[2].legend(fontsize=9, loc='upper right'); axs[2].set_title('Paredes fantasma', fontsize=11); axs[2].yaxis.set_major_formatter(COMA)
fig.legend(handles=[Patch(fc='black', label='ocupada'), Patch(fc='white', ec='#999', label='libre'), Patch(fc='#808080', label='sin ver (p = 0,5)'),
                    Patch(fc=ROJO, label='ocupada a más de 10 cm de una pared: fantasma')], fontsize=9, loc='lower center', ncol=4, frameon=False)
plt.tight_layout(rect=(0, 0.09, 1, 1)); plt.show()

---

## Soluciones

**Ejercicio 1.** Quedan 103 medidas de 182 y se ven las diez balizas, pero peor repartidas: el mapa pasa de 9,7 a 18,6 cm de error medio (de 17,8 a 32,0 de máximo) y la pose, de 13,0 a 33,9 cm de RMSE (máximo 57,8). Lo que no se mide, se adivina con la odometría.

**Ejercicio 2.** Con σ ×10, el último nodo queda a 3,7 cm, pero el máximo sube de 27 a 64,5 cm y el medio, de 12 a 37: el cierre pesa menos y reparte menos. Con ×100, casi no corrige: 47,9 cm al final y 93 de máximo, como la odometría sola. Cuánto se fía el grafo de una arista lo decide su covarianza.

**Ejercicio 3.** Apenas cambian: con la odometría, el 62 % (323 de 517) frente al 66 %; con el grafo, el 12 % (71 de 616) frente al 11 %. Bien clasificadas, 83,6 % y 87,8 %, frente a 83,3 % y 87,4 %. Las paredes dobles las quita el cierre de bucle, no los log-odds.

---

## Para llevarse de esta sesión

- **El SLAM**: mapa y pose a la vez; lo cierra volver a ver algo conocido.
- **EKF-SLAM**: todo se corrige junto, pero Σ crece con N².
- **La asociación**: una medida mal atribuida se queda en el mapa; la puerta la descarta, pero las buenas que rechaza crean copias.
- **El grafo**: el cierre reparte la deriva; a un cierre falso lo delata la suma.
- **La rejilla**: con las poses del grafo, el mapa de S22.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*